# Instituto Superior de Engenharia do Porto (ISEP)
## Licenciatura em Engenharia Física Aplicada (LEFA)
### Mecânica e Relatividade (MECREL) — 2026/2027
---
# Notebook TP05: Dinâmica da Partícula II — Máquinas de Atwood e Velocidade Terminal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristovaoDias/LEFA-MECREL/blob/main/Notebooks_Colab/TP05_Sistemas_Acoplados_Curvas_Arrasto_Colab.ipynb)

**Docente:** Cristóvão de Sousa Dias (`css@isep.ipp.pt`)  
**Objetivo Pedagógico:**
1. Simular numericamente o movimento de corpos acoplados (máquinas de Atwood com atrito em planos inclinados).
2. Comparar a aproximação assintótica à velocidade terminal em meios fluidos (vácuo vs. arrasto linear de Stokes vs. arrasto quadrático no ar).
3. **Auditoria GenAI (E.1):** Simular a aceleração em curva sobrelevada (Curva 3 de Zandvoort), auditar a *trajetória fantasma* gerada por presunção cinemática do LLM e determinar o instante e velocidade crítica de despiste.
4. **Engenharia de Prompts de Raiz (E.2):** Conceber um prompt de raiz para dimensionar o pacote aerodinâmico ($(C_L \cdot A)_{\min}$) que garanta a estabilidade dinâmica do monolugar sem derrapagem.

## 1. Setup & Configuração Científica

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuração do layout gráfico institucional do ISEP
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.35
print("✓ Ambiente configurado com sucesso!")

## 2. Modelo Físico 1: Queda Livre em Meios Resistentes (Vácuo vs. Stokes vs. Newton)

Para um corpo de massa $m$ largado do repouso, a 2.ª Lei de Newton fornece:
- **Queda no Vácuo:** $m \frac{dv}{dt} = mg \implies v(t) = gt$
- **Arrasto Linear Viscoso (Stokes):** $m \frac{dv}{dt} = mg - bv \implies v(t) = v_T \left(1 - e^{-t/\tau}\right), \quad v_T = \frac{mg}{b}, \; \tau = \frac{m}{b}$
- **Arrasto Quadrático no Ar (Newton):** $m \frac{dv}{dt} = mg - \frac{1}{2}C_D \rho A v^2 \implies v_T = \sqrt{\frac{2mg}{C_D \rho A}}$

Vamos explorar numericamente e graficamente a evolução temporal de cada modelo:

In [ ]:
# Parâmetros de simulação de referência
g = 9.80          # m/s^2
m = 80.0          # kg (paraquedista com fato)
v_T = 50.0        # m/s (velocidade terminal alvo)
tau = v_T / g     # constante de tempo característica (s)
b = m / tau       # coeficiente linear equivalente (N*s/m)

t = np.linspace(0, 25, 500)

# Soluções analíticas
v_vacuo = g * t
v_linear = v_T * (1 - np.exp(-t / tau))
v_quadratico = v_T * np.tanh(g * t / v_T)

# Visualização gráfica comparativa
plt.figure(figsize=(10, 5))
plt.plot(t, v_vacuo, '--', color='#EF4444', lw=2, label='Vácuo ($v = gt$)')
plt.plot(t, v_linear, '-', color='#0077B6', lw=2.5, label=f'Arrasto Linear (Stokes, $\\tau = {tau:.1f}\\text{{ s}}$)')
plt.plot(t, v_quadratico, '-.', color='#10B981', lw=2.5, label='Arrasto Quadrático (Ar)')
plt.axhline(v_T, color='#F59E0B', lw=1.8, ls=':', label=f'Velocidade Terminal $v_T = {v_T:.1f}\\text{{ m/s}}$')

plt.title('Dinâmica de Queda: Vácuo vs. Fluidos Viscosos e Aerodinâmica', fontweight='bold', color='#0A2540')
plt.xlabel('Tempo $t$ (s)', fontweight='bold')
plt.ylabel('Velocidade de Queda $v(t)$ (m/s)', fontweight='bold')
plt.ylim(0, 65)
plt.legend(framealpha=0.9)
plt.show()

## 3. Modelo Físico 2: Simulação Numérica de Máquinas de Atwood com Atrito

Para um sistema de dois corpos ligados por fio com bloco $m_1$ sobre rampa $\theta$ com atrito e $m_2$ suspenso:
$$a = \frac{m_2 g - m_1 g \sin\theta - f_a}{m_1 + m_2}$$
Execute o bloco abaixo para explorar a aceleração e a tensão no fio para diferentes valores de $m_2$ e $\mu_k$:

In [ ]:
# Parâmetros do sistema acoplado
m1 = 6.0    # kg
theta_deg = 35.0
theta = np.radians(theta_deg)
mu_s = 0.30
mu_k = 0.20

N1 = m1 * g * np.cos(theta)
P1x = m1 * g * np.sin(theta)
fs_max = mu_s * N1

m2_array = np.linspace(0.5, 12.0, 100)
a_array = []
T_array = []

for m2 in m2_array:
    P2 = m2 * g
    if P2 > P1x + fs_max:
        # Acelera rampa acima
        fk = mu_k * N1
        a = (P2 - P1x - fk) / (m1 + m2)
        T = m2 * (g - a)
    elif P2 < P1x - fs_max:
        # Acelera rampa abaixo
        fk = mu_k * N1
        a = (P1x - fk - P2) / (m1 + m2)
        T = m2 * (g + a)
    else:
        # Repouso estático
        a = 0.0
        T = P2
    a_array.append(a)
    T_array.append(T)

plt.figure(figsize=(10, 4.5))
plt.plot(m2_array, a_array, color='#0077B6', lw=2.5, label='Aceleração $a$ (m/s²)')
plt.axvspan((P1x - fs_max)/g, (P1x + fs_max)/g, color='#FEF3C7', alpha=0.5, label='Zona de Repouso Estático ($a = 0$)')
plt.title('Resposta Dinâmica da Máquina de Atwood em Rampa de 35°', fontweight='bold', color='#0A2540')
plt.xlabel('Massa Suspensa $m_2$ (kg)', fontweight='bold')
plt.ylabel('Aceleração $a$ (m/s²)', fontweight='bold')
plt.legend()
plt.show()

---
## 4. Desafio GenAI & Auditoria Física (Parte E da Ficha TP05)

Na formação de Engenharia Física, a Inteligência Artificial Generativa deve ser sujeita a um processo contínuo de **auditoria científica**. Esta secção está dividida em dois exercícios:
1. **Exercício E.1 — Auditoria à Trajetória Fantasma:** Submeter um prompt com ambiguidade física, demonstrar que o LLM gera um código sem erros de sintaxe mas com uma violação física catastrófica (o carro derrapa e despista-se aos $t \approx 2.05\text{ s}$).
2. **Exercício E.2 — Engenharia de Prompts de Raiz:** Sem qualquer prompt fornecido, conceber um prompt estruturado para dimensionar o pacote aerodinâmico ($(C_L \cdot A)_{\min}$) que viabilize a manobra sem despiste.

### 4.1 Exercício E.1: O Paradoxo da Trajetória Fantasma (Auditoria Crítica)

> **Instruções:**
> 1. Copie o prompt fornecido no Exercício E.1 da Ficha TP05 e submeta-o ao seu modelo de IA (ChatGPT, Claude, Gemini ou Copilot).
> 2. Cole o código gerado pelo modelo na **Célula E.1A** abaixo e execute-o.
> 3. Preencha o **Relatório de Auditoria** na célula de texto seguinte.
> 4. Na **Célula E.1B**, desenvolva o script de verificação do rácio de saturação de atrito $\eta(t)$ e identifique numericamente o instante de despiste $t_{\text{despiste}}$.

#### Célula E.1A: Código Gerado pelo LLM a partir do Prompt Ambíguo (Cole aqui o código bruto da IA)

In [ ]:
# [ESPAÇO RESERVADO PARA O ESTUDANTE]
# Cole aqui o código completo gerado pelo LLM a partir do prompt ambíguo de E.1 e execute.


#### Relatório da Auditoria Física de E.1 pelo Estudante

*(Preencha os campos abaixo com as suas conclusões de auditoria científica:)*

- **O script do LLM executou sem erros de programação?** `[Sim / Não]`
- **Velocidade final $v(4)$ e aceleração normal $a_n(4)$ previstas pela IA:** `v = _____ m/s (_____ km/h)`, `a_n = _____ m/s² (_____ g)`
- **Diagnóstico da 'Trajetória Fantasma':** *(Explique porque é que os gráficos gerados pela IA são fisicamente enganadores e em que instante a lei de Coulomb é violada)*
- **Instante crítico de despiste ($t_{\text{despiste}}$):** `_____ s`
- **Velocidade crítica de despiste ($v_{\text{despiste}}$):** `_____ m/s (_____ km/h)`
- **Conclusão Epistemológica:** *(Porque é que a IA gerou curvas suaves sem avisar o utilizador de que o carro colidiria contra os rails?)*

#### Célula E.1B: Auditoria Computacional — Verificação de Aderência $\eta(t)$ e Instante de Despiste

In [ ]:
# [ESPAÇO RESERVADO PARA O ESTUDANTE]
# Implemente o cálculo do atrito tangencial (ft = m * at), atrito lateral (f_lat) e atrito máximo (f_max = mu_s * N).
# Calcule o rácio eta(t) = f_req(t) / f_max(t) e trace o gráfico de eta(t) ao longo dos 4 segundos.
# Identifique o ponto onde eta(t) > 1.0 (instante de despiste).


---
### 4.2 Exercício E.2: Engenharia de Prompts de Raiz (Dimensionamento Aerodinâmico)

> **Instruções:**
> 1. **Não use nenhum prompt pronto.** Formule de raiz o seu próprio prompt estruturado de engenharia para dimensionar $(C_L \cdot A)_{\min}$ tal que $\eta(t) \le 1.0$ para todo o $t \in [0, 4.0\text{ s}]$.
> 2. Cole o texto integral do seu prompt na célula de texto **Célula E.2A**.
> 3. Cole o código gerado pelo modelo na **Célula E.2B** e execute-o.
> 4. Preencha a auditoria final na **Célula E.2C**.

#### Célula E.2A: O Seu Prompt de Engenharia Concebido de Raiz

> *(Cole aqui o texto exato do prompt que redigiu para submeter ao LLM)*

#### Célula E.2B: Código Gerado pelo LLM a partir do seu Prompt de Raiz

In [ ]:
# [ESPAÇO RESERVADO PARA O ESTUDANTE]
# Cole aqui o código gerado pela IA com a inclusão do pacote aerodinâmico e o dimensionamento de (CL * A)_min.


#### Célula E.2C: Relatório de Auditoria ao Prompt de Raiz

*(Preencha os campos com a avaliação do desempenho do LLM:)*

- **Valor de $(C_L \cdot A)_{\min}$ calculado pela IA:** `_____ m²`
- **Valor da sua dedução analítica de referência:** `_____ m²`
- **Auditoria Vetorial ao Downforce:** *(A IA considerou o downforce perpendicular à pista inclinada ou vertical para a Terra? Houve erro na projeção de forças?)*
- **Auditoria ao Círculo de Atrito:** *(O modelo combinou quadraticamente o atrito tangencial motriz com o atrito centrípeta lateral?)*